# Annotation Coverage

Who annotated which items, where annotators overlap, and what is still open.

All three tasks are served from the same 1072-item queue (`corpus.parquet`, in
the same order as the CSV potato serves), and **the trailing integer of
`safe_instance_id` is that item's position in the queue** — so "index" here means
a number in `0..1071` that maps straight onto what an annotator sees.

- **Annotated** means `annotation_order_{annotator}` is non-null for that item.
- **Open** means *nobody* has annotated it on that task — an index absent from the
  task parquet. Items only the adjudicator rated are not open (they are covered,
  just not double-annotated); those show up in the overlap section instead.

`tejo9855` is the adjudicator, so `_gold` duplicates `_tejo9855` and `gold` is not
reported as a separate annotator.

In [ ]:
import matplotlib as mpl
import numpy as np
import pandas as pd
from IPython.display import display

from nb_utils import (
    TASKS, load, DIMENSIONS, setup_plots,
    SERIES, BLUE_RAMP, SURFACE, INK, INK_2, INK_MUTED, GRID,
    annotation_index, coverage, overlap, items_by_annotator_count,
    open_indexes, as_ranges, format_ranges,
)

plt = setup_plots(minimal=True)

UNIVERSE = len(load('corpus'))
ann_idx = annotation_index()

TASK_LABEL = {
    'setting':        'Setting',
    'agency':         'Agency',
    'event_relation': 'Event Relation',
}

print(f'queue: {UNIVERSE} items (indexes 0-{UNIVERSE - 1})\n')
for t in TASKS:
    touched = ann_idx.loc[ann_idx.task == t, 'idx'].nunique()
    print(f'{TASK_LABEL[t]:15s} {touched:4d} touched   {UNIVERSE - touched:4d} open'
          f'   {ann_idx.loc[ann_idx.task == t, "annotator"].nunique()} annotators')
print(f'\nuntouched on every task: {len(open_indexes())}')

In [ ]:
# Annotators are ranked by total work so the busiest takes the first
# categorical slot, and the mapping is frozen here -- a person keeps the
# same color in every figure below.
_totals = ann_idx.groupby('annotator').size().sort_values(ascending=False)
ANNOTATORS = list(_totals.index)
COLOR = {a: SERIES[i % len(SERIES)] for i, a in enumerate(ANNOTATORS)}
print(pd.DataFrame({'items (all tasks)': _totals, 'color': pd.Series(COLOR)}).to_string())

## 1. What each annotator covered

`n_skipped_in_range` counts indexes an annotator passed over *inside* the block
they worked — distinct from indexes they simply never reached.
`n_complete_all_dims` counts items where every dimension of the task was
answered, which matters for Event Relation: many items have the span-is-event
questions answered but temporal order and causality left blank.

In [ ]:
cov = coverage()
for t in TASKS:
    sub = cov[cov.task == t].drop(columns='task')
    print(f'\n{TASK_LABEL[t]}  ({len(DIMENSIONS[t])} dimensions)')
    print(sub.to_string(index=False))

### The exact index runs

The blocks each annotator actually worked. Use `annotation_index()` for the
per-item rows behind this (one row per annotator per item, with their queue
position and how many dimensions they filled in).

In [ ]:
for t in TASKS:
    print(f'\n{TASK_LABEL[t]}')
    sub = ann_idx[ann_idx.task == t]
    for ann in [a for a in ANNOTATORS if a in set(sub.annotator)]:
        idxs = sorted(sub.loc[sub.annotator == ann, 'idx'])
        print(f'  {ann:16s} {len(idxs):4d} items   {format_ranges(idxs)}')

In [ ]:
# Every index one annotator did, as a plain list -- change these two names.
who, task = 'roda9210', 'setting'
mine = sorted(ann_idx.loc[(ann_idx.annotator == who) & (ann_idx.task == task), 'idx'])
print(f'{who} on {task}: {len(mine)} items')
print(mine)

## 2. Coverage map

One lane per annotator, drawn across the whole queue. Filled segments are
annotated indexes; the pale track behind each lane is the rest of the queue.
The dashed rule marks the furthest index anyone has reached on that task —
everything to its right is untouched by everyone.

In [ ]:
fig, axes = plt.subplots(
    len(TASKS), 1, figsize=(13, 7), sharex=True,
    gridspec_kw={'height_ratios': [ann_idx[ann_idx.task == t].annotator.nunique()
                                   for t in TASKS], 'hspace': 0.35},
)

for ax, t in zip(axes, TASKS):
    sub = ann_idx[ann_idx.task == t]
    lanes = [a for a in ANNOTATORS if a in set(sub.annotator)]
    lanes = sorted(lanes, key=lambda a: -(sub.annotator == a).sum())
    frontier = int(sub.idx.max()) + 1

    for row, ann in enumerate(lanes):
        y = len(lanes) - row - 1
        idxs = sorted(sub.loc[sub.annotator == ann, 'idx'])
        ax.broken_barh([(0, UNIVERSE)], (y - 0.3, 0.6),
                       facecolors='#f2f1ed', edgecolors='none', zorder=1)
        # +1 so a single-index run is still one full queue slot wide.
        # antialiased=False: at ~1px per index, feathered edges would smear a
        # one-index skip into a gap several times its true width.
        runs = [(a, b - a + 1) for a, b in as_ranges(idxs)]
        ax.broken_barh(runs, (y - 0.3, 0.6), facecolors=COLOR[ann],
                       edgecolors='none', antialiased=False, zorder=2)
        ax.text(UNIVERSE + 12, y, f'{len(idxs)}', va='center', ha='left',
                fontsize=9, color=INK_2)

    ax.axvline(frontier, color=INK_MUTED, ls=(0, (4, 3)), lw=1, zorder=3)
    ax.text(frontier + 12, len(lanes) - 0.45, f'reached {frontier}',
            fontsize=8.5, color=INK_MUTED, va='top')

    ax.set_yticks(range(len(lanes)))
    ax.set_yticklabels(lanes[::-1], fontsize=9.5)
    ax.set_ylim(-0.7, len(lanes) - 0.3)
    ax.set_xlim(0, UNIVERSE + 130)
    ax.tick_params(length=0)
    ax.set_title(f'{TASK_LABEL[t]}   ·   {sub.idx.nunique()} of {UNIVERSE} items covered',
                 loc='left', color=INK, pad=6)

axes[-1].set_xlabel('queue index')
axes[-1].set_xticks(list(range(0, UNIVERSE, 100)) + [UNIVERSE - 1])
fig.suptitle('Which indexes each annotator has done', x=0.125, ha='left',
             fontsize=14, color=INK, y=0.98)
plt.show()

## 3. Overlap

How many indexes each pair of annotators both did — the material available for
inter-annotator agreement. The diagonal is that person's own total and is shaded
gray, because it is not an overlap and would otherwise dominate the color scale.

In [ ]:
cmap = mpl.colors.LinearSegmentedColormap.from_list('blues', BLUE_RAMP)

_sizes = [len(overlap(t)) for t in TASKS]
fig, axes = plt.subplots(1, len(TASKS), figsize=(14, 4.2),
                         gridspec_kw={'wspace': 0.45, 'width_ratios': _sizes})

for ax, t in zip(axes, TASKS):
    m = overlap(t)
    off = m.where(~np.eye(len(m), dtype=bool))
    vmax = max(off.max().max(), 1)
    ax.imshow(off.to_numpy(), cmap=cmap, vmin=0, vmax=vmax)

    for i in range(len(m)):
        for j in range(len(m)):
            v = int(m.iat[i, j])
            if i == j:
                ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1,
                                           facecolor='#eceae4', edgecolor='none'))
                ax.text(j, i, v, ha='center', va='center', fontsize=9,
                        color=INK_MUTED)
            else:
                shade = v / vmax
                ax.text(j, i, v, ha='center', va='center', fontsize=9,
                        color='#ffffff' if shade > 0.55 else INK)
    # 2px surface gap between cells
    ax.set_xticks(np.arange(-0.5, len(m), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(m), 1), minor=True)
    ax.grid(which='minor', color=SURFACE, lw=2)
    ax.tick_params(which='minor', length=0)

    ax.set_xticks(range(len(m)))
    ax.set_yticks(range(len(m)))
    ax.set_xticklabels(m.columns, rotation=45, ha='right', fontsize=8.5)
    ax.set_yticklabels(m.index, fontsize=8.5)
    ax.tick_params(length=0)
    ax.set_anchor('N')   # equal aspect centers each panel; pin them to a common top
    ax.set_title(TASK_LABEL[t], loc='left', color=INK, pad=8)

fig.suptitle('Shared indexes per annotator pair', x=0.125, ha='left',
             fontsize=14, color=INK, y=1.02)
plt.show()

for t in TASKS:
    print(f'\n{TASK_LABEL[t]}')
    print(overlap(t).to_string())

### How much is multiply annotated

Items carrying one annotator have no agreement signal; two or more do.

In [ ]:
_bars = [len(items_by_annotator_count(t)) for t in TASKS]
fig, axes = plt.subplots(1, len(TASKS), figsize=(13, 3.4), sharey=True,
                         layout='constrained',
                         gridspec_kw={'wspace': 0.12, 'width_ratios': _bars})

for ax, t in zip(axes, TASKS):
    counts = items_by_annotator_count(t)
    ax.bar(counts.index, counts.to_numpy(), width=0.55, color=SERIES[0])
    for x, y in counts.items():
        ax.text(x, y + 6, f'{y}', ha='center', va='bottom', fontsize=9, color=INK_2)
    n_multi = int(counts[counts.index >= 2].sum())
    ax.set_title(f'{TASK_LABEL[t]}  ·  {n_multi} multiply annotated',
                 loc='left', color=INK, pad=6, fontsize=11)
    ax.set_xticks(counts.index)
    ax.set_xlabel('annotators on the item')
    ax.grid(axis='y', zorder=0)
    ax.set_axisbelow(True)
    ax.tick_params(length=0)

axes[0].set_ylabel('items')
axes[0].set_ylim(0, max(items_by_annotator_count(t).max() for t in TASKS) * 1.12)
plt.show()

## 4. What is open

Indexes nobody has annotated on that task. Setting and Agency are a clean
frontier — everything past the last adjudicated item. Event Relation also has
holes *inside* the worked region, where items were skipped rather than rated.

In [ ]:
for t in TASKS:
    op = open_indexes(t)
    runs = as_ranges(op)
    interior = [r for r in runs if r[1] < max(ann_idx[ann_idx.task == t].idx)]
    print(f'\n{TASK_LABEL[t]}: {len(op)} open of {UNIVERSE}')
    print(f'  ranges:      {format_ranges(op)}')
    print(f'  holes inside the worked region: '
          f'{sum(b - a + 1 for a, b in interior)} items in {len(interior)} runs'
          + (f' -> {format_ranges([i for a, b in interior for i in range(a, b + 1)])}'
             if interior else ''))

everywhere = open_indexes()
print(f'\nuntouched on all three tasks: {len(everywhere)} items -> {format_ranges(everywhere)}')

In [ ]:
# Assertion: open + covered must account for the whole queue on every task.
for t in TASKS:
    covered = ann_idx.loc[ann_idx.task == t, 'idx'].nunique()
    assert len(open_indexes(t)) + covered == UNIVERSE, t
    assert covered == len(load(t)), t
print('coverage reconciles with the queue on all three tasks')

## 5. Where the next annotator should start

For fresh coverage, start at the first open index. For agreement, start at an
index the adjudicator has already done but no one else has — those are listed
separately below.

In [ ]:
for t in TASKS:
    op = open_indexes(t)
    sub = ann_idx[ann_idx.task == t]
    solo = sorted(sub.groupby('idx').annotator.nunique().pipe(lambda s: s[s == 1]).index)
    print(f'\n{TASK_LABEL[t]}')
    print(f'  new coverage      -> start at index {op[0]} ({len(op)} open)')
    print(f'  double annotation -> {len(solo)} singly-annotated items, '
          f'first at {solo[0] if solo else "-"}')
    print(f'                       {format_ranges(solo, max_runs=6)}')